# Module 4: Behavioral Navigation, Quota Exploration Matrix, Unit Economics, Promo Parity & Paid CAC Bleed
**SQL Reference:** [`sql/04_economics_and_behavior.sql`](../sql/04_economics_and_behavior.sql) | **Visual Charts:** [`figures/05_behavioral_fatigue_and_indecision.png`](../figures/05_behavioral_fatigue_and_indecision.png), [`figures/plan_migration_matrix.png`](../figures/plan_migration_matrix.png), [`figures/lost_revenue_waterfall.png`](../figures/lost_revenue_waterfall.png), [`figures/07_unit_economics_and_promo_parity.png`](../figures/07_unit_economics_and_promo_parity.png)

## Executive Summary (`H4`, `H7`, `H8` Confirmed | `H5` Rejected)
1. **Golden 2–3 Event Window & Indecision Cliff (`H8` Confirmed):** **`92.57%` (`22,638 / 24,456`)** of all conversions occur within exactly **`2–3` events (`30.37%` CR)**. Browsing `3+` countries collapses engaged CR by **`-72.3%` (`24.11%` -> `6.69%`)**. Among `24,456` converting sessions, **`3,078` (`12.59%`)** cross-shopped an alternative destination before buying — led by **`EU <-> AT` (`693` sessions / `22.51%`)**, where European travelers compare regional `EU` vs. single-country `AT` (`$16.65` vs. `$9.79` for 5 GB).
2. **Quota Exploration Matrix (`H5` Rejected):** Across `24,426` converting sessions with event telemetry, **`92.16%` (`22,512`)** viewed only a single quota tier, and **`4.96%` (`1,211`)** explored multiple quotas and bought the **largest** quota viewed (`97.12%` / `23,723` on the diagonal vs. only `2.88%` / `703` off-diagonal).
3. **Unit Economics (`$/GB`) & Implied Promo Asymmetry (`H7` Confirmed):** `10 GB` plans offer a `39%` (`US`) to `56%` (`TR`) lower cost per GB than `1 GB` plans. Modal anchor decomposition across `24,782` orders (`98.6%` of volume) proves **`25.50%` of iOS orders receive below-anchor promo pricing vs. `21.07%` on Compatible Android (`+4.43 pp` iOS promo advantage)**, yielding **`+$39k` to `+$134k/yr`** in Android parity upside.
4. **Paid CAC Bleed on Incompatible Android (`H4` Confirmed):** At a `$1.50` blended CAC (`$3.00` Paid CPA at 50% paid share), acquiring `27,073` incompatible sessions destroys **`-$39,371.31` net over 27 days (`-$532k/yr`)**.


In [1]:
import pandas as pd

# Verified BigQuery Output from sql/04_economics_and_behavior.sql (Query 4.1, 4.2, 4.3, 4.4)
fatigue_df = pd.DataFrame([
    {"Event_Depth_Tier": "0 Events (Bounce)", "Unique_Sessions": 125553, "Converting_Sessions": 29, "Session_CR_Pct": 0.02, "Share_of_All_Buyers_Pct": 0.12},
    {"Event_Depth_Tier": "1 Event (Abandoned S1)", "Unique_Sessions": 31305, "Converting_Sessions": 429, "Session_CR_Pct": 1.37, "Share_of_All_Buyers_Pct": 1.75},
    {"Event_Depth_Tier": "2-3 Events (Golden Window)", "Unique_Sessions": 74546, "Converting_Sessions": 22638, "Session_CR_Pct": 30.37, "Share_of_All_Buyers_Pct": 92.57},
    {"Event_Depth_Tier": "4-6 Events (Moderate Exploration)", "Unique_Sessions": 7960, "Converting_Sessions": 1231, "Session_CR_Pct": 15.46, "Share_of_All_Buyers_Pct": 5.03},
    {"Event_Depth_Tier": "7+ Events (Severe Search Fatigue)", "Unique_Sessions": 2184, "Converting_Sessions": 129, "Session_CR_Pct": 5.91, "Share_of_All_Buyers_Pct": 0.53}
])
print("Event Depth & Search Fatigue:")
print(fatigue_df.to_string(index=False))

quota_summary_df = pd.DataFrame([
    {"Exploration_Behavior": "1. Single Quota Viewed & Bought", "Converting_Sessions": 22512, "Share_Pct": 92.16},
    {"Exploration_Behavior": "2. Multi-Quota Viewed -> Bought Largest Viewed", "Converting_Sessions": 1211, "Share_Pct": 4.96},
    {"Exploration_Behavior": "3. Multi-Quota Viewed -> Bought Smallest Viewed", "Converting_Sessions": 668, "Share_Pct": 2.73},
    {"Exploration_Behavior": "4. Multi-Quota Viewed -> Bought Middle/Other", "Converting_Sessions": 35, "Share_Pct": 0.14}
])
print("\nTimestamp-Free Quota Exploration Summary (N = 24,426):")
print(quota_summary_df.to_string(index=False))


Event Depth & Search Fatigue:
                 Event_Depth_Tier  Unique_Sessions  Converting_Sessions  Session_CR_Pct  Share_of_All_Buyers_Pct
                0 Events (Bounce)           125553                   29            0.02                     0.12
           1 Event (Abandoned S1)            31305                  429            1.37                     1.75
       2-3 Events (Golden Window)            74546                22638           30.37                    92.57
4-6 Events (Moderate Exploration)             7960                 1231           15.46                     5.03
7+ Events (Severe Search Fatigue)             2184                  129            5.91                     0.53

Timestamp-Free Quota Exploration Summary (N = 24,426):
                           Exploration_Behavior  Converting_Sessions  Share_Pct
                1. Single Quota Viewed & Bought                22512      92.16
 2. Multi-Quota Viewed -> Bought Largest Viewed                 1211       4

In [2]:
# Verified BigQuery Output from sql/04_economics_and_behavior.sql (Query 4.6, 4.7, 4.8): Promo Asymmetry & CAC Bleed
promo_df = pd.DataFrame([
    {"Cohort": "iOS (not_applicable)", "Evaluated_Orders": 16244, "Modal_List_Revenue": 256674.06, "Actual_Realized_Revenue": 235771.09, "Discounted_Orders": 4142, "Implied_Promo_Share_Pct": 25.50, "Avg_Concession_Per_Order": 1.29},
    {"Cohort": "Android Compatible (true)", "Evaluated_Orders": 8428, "Modal_List_Revenue": 124667.72, "Actual_Realized_Revenue": 115058.13, "Discounted_Orders": 1776, "Implied_Promo_Share_Pct": 21.07, "Avg_Concession_Per_Order": 1.14},
    {"Cohort": "Android Incompatible (false)", "Evaluated_Orders": 110, "Modal_List_Revenue": 1351.90, "Actual_Realized_Revenue": 1207.20, "Discounted_Orders": 29, "Implied_Promo_Share_Pct": 26.36, "Avg_Concession_Per_Order": 1.32}
])
print("Implied Promotional Discount Decomposition (H7):")
print(promo_df.to_string(index=False))

cac_df = pd.DataFrame([
    {"Scenario": "Low ($0.75 Blended CAC / $1.50 Paid CPA)", "Gross_Spend_27d": 20304.75, "Realized_Rev_27d": 1238.19, "Net_Loss_27d": -19066.56, "Annualized_Bleed": -257752},
    {"Scenario": "Base ($1.50 Blended CAC / $3.00 Paid CPA)", "Gross_Spend_27d": 40609.50, "Realized_Rev_27d": 1238.19, "Net_Loss_27d": -39371.31, "Annualized_Bleed": -532241},
    {"Scenario": "Mid-High ($2.50 Blended CAC / $5.00 Paid CPA)", "Gross_Spend_27d": 67682.50, "Realized_Rev_27d": 1238.19, "Net_Loss_27d": -66444.31, "Annualized_Bleed": -898229},
    {"Scenario": "High ($3.50 Blended CAC / $7.00 Paid CPA)", "Gross_Spend_27d": 94755.50, "Realized_Rev_27d": 1238.19, "Net_Loss_27d": -93517.31, "Annualized_Bleed": -1264216}
])
print("\nPaid CAC Bleed Sensitivity Across 27,073 Incompatible Sessions (H4):")
print(cac_df.to_string(index=False))


Implied Promotional Discount Decomposition (H7):
                      Cohort  Evaluated_Orders  Modal_List_Revenue  Actual_Realized_Revenue  Discounted_Orders  Implied_Promo_Share_Pct  Avg_Concession_Per_Order
        iOS (not_applicable)             16244           256674.06                235771.09               4142                    25.50                      1.29
   Android Compatible (true)              8428           124667.72                115058.13               1776                    21.07                      1.14
Android Incompatible (false)               110             1351.90                  1207.20                 29                    26.36                      1.32

Paid CAC Bleed Sensitivity Across 27,073 Incompatible Sessions (H4):
                                     Scenario  Gross_Spend_27d  Realized_Rev_27d  Net_Loss_27d  Annualized_Bleed
     Low ($0.75 Blended CAC / $1.50 Paid CPA)         20304.75           1238.19     -19066.56           -257752
    Bas